# CVI Forecasting Dashboard on Google Colab



**Before you start:** upload `CVI_dashboard_bundle.zip` (about 170 MB) to your Google Drive.

## 1. Get the bundle

In [2]:
import os, zipfile

USE_DRIVE = True
BUNDLE = '/content/drive/MyDrive/CVI_dashboard_bundle.zip'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    from google.colab import files
    uploaded = files.upload()
    BUNDLE = '/content/' + next(iter(uploaded))

zipfile.ZipFile(BUNDLE).extractall('/content')
os.chdir('/content/cvi_dashboard')
print('Bundle ready:', sorted(os.listdir('.')))

Mounted at /content/drive
Bundle ready: ['cvi_outputs', 'dashboard']


## 2. Install packages

In [3]:
!pip -q install -U streamlit plotly lightgbm pyarrow

## 3. Start the dashboard (runs in the background)

In [4]:
import subprocess, time, urllib.request
server = subprocess.Popen(['streamlit', 'run', 'dashboard/app.py', '--server.port', '8501', '--server.headless', 'true',
                           '--browser.gatherUsageStats', 'false'], stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8501/_stcore/health', timeout=2)
        print('Streamlit is running on port 8501'); break
    except Exception:
        time.sleep(2)
else:
    print(open('streamlit.log').read())

Streamlit is running on port 8501


## 4. Get a public link



In [5]:
import re, subprocess
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(200):
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tunnel.stdout.readline())
    if m:
        url = m.group(0); break
print('\n>>> Open the dashboard here:', url, '\n' if url else '(no URL yet - re-run this cell)')


>>> Open the dashboard here: https://impose-thumbnail-postposted-cooking.trycloudflare.com 

